In [1]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import plotly.express as px

In [2]:
results_dir = Path("/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results")
parquet_files = list(results_dir.rglob("*last.parquet"))

dfs = []
for f in tqdm(parquet_files):
    df = pd.read_parquet(f,columns=["generated_text",'finish_reason','options','extracted','prediction'])
    df['model'] = f.parent.name[20:]
    df['benchmark'] = f.parent.parent.name
    dfs.append(df)

ans = pd.concat(dfs, ignore_index=True)

del dfs

100%|██████████| 84/84 [00:18<00:00,  4.47it/s]


In [4]:
npm = pd.read_json('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/test_np_pet/2025-08-21T17:56:29-Qwen2.5-3B-DrGRPO-Stratified/test_np_pet_output.jsonl',lines=True)

In [5]:
npm

""


In [3]:
df = ans.value_counts(subset=['benchmark','model','finish_reason']).to_frame().reset_index()
df[df['finish_reason'] == 'length'].sort_values(['count'])

,benchmark,model,finish_reason,count
99,medexpqa,Qwen3-4B,length,1
97,medqa_test,Qwen2.5-7B-Instruct,length,1
98,medmcqa,Qwen2.5-7B-Instruct,length,1
100,test_cog,Qwen2.5-7B-Instruct,length,1
96,clinical_knowledge,Qwen3-4B,length,2
94,USMLE_step1,Llama-3.2-3B-Instruct,length,2
95,USMLE_step1,Qwen3-0.6B,length,2
93,USMLE_step1,Qwen3-4B,length,3
92,medqa_test,Qwen3-4B,length,4
91,medqa_test,Qwen3-0.6B,length,4


In [4]:
invalid_counts = ans[ans['extracted'] == 'invalid'].groupby(['model','benchmark']).size().rename('invalid_count').to_frame().reset_index()

In [5]:
invalid_counts_recovered = ans[ans['prediction'] == 'invalid'].groupby(['model','benchmark']).size().rename('invalid_count').to_frame().reset_index()

- Llama 3.2-3B still cannot recover a lot of answers
- All models struggle with recovering medmcqa, are those answers ambiguous? can we filter them?

In [6]:
px.bar(
    invalid_counts_recovered,
    x='invalid_count',
    y='model',
    color='benchmark',
    height=500,
    title='Number of invalid answers after LLM recovery'
)

In [7]:
px.bar(
    invalid_counts,
    x='invalid_count',
    y='model',
    color='benchmark',
    height=500,
    title='Number of invalid answers after naive regex extraction'
)

In [10]:
ans[
    # (ans['model'] == 'HuatuoGPT-o1-8B') &
    (ans['prediction'] == 'invalid') &
    (ans['benchmark'] == 'medmcqa')
].sample(5)

,generated_text,finish_reason,options,extracted,prediction,model,benchmark
65460,To answer the question regarding which disorde...,stop,1. MEN syndrome 2. Cyanotic heart disease 3. A...,invalid,invalid,Qwen2.5-3B-DrGRPO,medmcqa
99315,"To answer this question, let's consider the ch...",stop,1. CPK-MB 2. SGOT (AST) 3. LDH 4. All of the a...,invalid,invalid,Llama-3.2-3B-Instruct,medmcqa
21229,To determine what treatment is appropriate for...,stop,1. Chloroquine 2. Hydroxycholoquine 3. Dexamet...,invalid,invalid,Qwen2.5-3B-Instruct,medmcqa
29340,To determine which of the given options (Olmes...,stop,1. Olmesaan 2. Candesaan 3. Telmisaan 4. Eprosaan,invalid,invalid,Qwen2.5-3B-Instruct,medmcqa
47612,To accurately evaluate each statement regardin...,stop,"1. a is false and b, c, d are true 2. a, b are...",invalid,invalid,Qwen2.5-7B-Instruct,medmcqa


In [11]:
print(ans.loc[65460].generated_text)

To answer the question regarding which disorder is represented by the phrase 'Blueberry lips', it is essential to understand the context. Blueberry lips is a common colloquial term that refers to hemangiomas, localized clusters of blood vessels that appear as raised, reddish or purplish colored patches. In childhood, hemangiomas are very common and usually fade over time, but they can sometimes be paler due to lack of blood, giving them a 'blueberry' appearance.

Now we need to review the provided medical condition alternatives:

1. **MEN syndrome (Multiple Endocrine Neoplasia syndrome)**: This is a group of inherited disorders in which various endocrine glands may develop tumors. It does not describe 'Blueberry lips'.

2. **Cyanotic heart disease**: This is a group of heart conditions that result in a blue or purple-blue discoloration of the skin and mucous membranes (cyanosis), typically due to inadequate amounts of oxygen in the blood. It is a condition, not a syndrome, and does not